In [ ]:
import pandas as pd
import spacy
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv('/content/Emotion_classify_Data.csv')

In [ ]:
df

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear
...,...,...
5932,i begun to feel distressed for you,fear
5933,i left feeling annoyed and angry thinking that...,anger
5934,i were to ever get married i d have everything...,joy
5935,i feel reluctant in applying there because i w...,fear


In [ ]:
nlp = spacy.load('en_core_web_sm')

In [ ]:
def preprocessed_text(raw_texts):
  processed_texts = []
  for i in raw_texts:
    doc = nlp(i)
    processed_texts.append(' '.join([token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.is_space]))
  return processed_texts

In [ ]:
df['processed_texts'] = preprocessed_text(df['Comment'])

In [ ]:
le = LabelEncoder()
df['Encoded_Emotions'] = le.fit_transform(df['Emotion'])


In [ ]:
df

,Comment,Emotion,Encoded_Emotions
0,i seriously hate one subject to death but now ...,fear,1
1,im so full of life i feel appalled,anger,0
2,i sit here to write i start to dig out my feel...,fear,1
3,ive been really angry with r and i feel like a...,joy,2
4,i feel suspicious if there is no one outside l...,fear,1
...,...,...,...
5932,i begun to feel distressed for you,fear,1
5933,i left feeling annoyed and angry thinking that...,anger,0
5934,i were to ever get married i d have everything...,joy,2
5935,i feel reluctant in applying there because i w...,fear,1


In [ ]:
df['processed_texts'] = preprocessed_text(df['Comment'])
x_train, x_test, y_train, y_test = train_test_split(df['processed_texts'], df ['Encoded_Emotions'], test_size=0.2, random_state=42)

In [ ]:
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [39]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

# Create a pipeline with TfidfVectorizer and XGBClassifier
model = Pipeline([
    ('tfidf', TfidfVectorizer()), # Step 1: Text vectorization
    ('xgb', XGBClassifier())      # Step 2: XGBoost Classifier
])


param_grid = {
    'xgb__max_depth': [3, 5, 7],
    'xgb__learning_rate': [0.01, 0.1, 0.2],
    'xgb__n_estimators': [100, 200]
}

grid = GridSearchCV(model, param_grid, cv=5)
grid.fit(x_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                       ('xgb',
                                        XGBClassifier(base_score=None,
                                                      booster=None,
                                                      callbacks=None,
                                                      colsample_bylevel=None,
                                                      colsample_bynode=None,
                                                      colsample_bytree=None,
                                                      device=None,
                                                      early_stopping_rounds=None,
                                                      enable_categorical=False,
                                                      eval_metric=None,
                                                      feature_types=None,
                                                      feature_weights=None,
                                                      gamma=None,
                                                      grow_policy=No...
                                                      max_bin=None,
                                                      max_cat_threshold=None,
                                                      max_cat_to_onehot=None,
                                                      max_delta_step=None,
                                                      max_depth=None,
                                                      max_leaves=None,
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=None,
                                                      num_parallel_tree=None, ...))]),
             param_grid={'xgb__learning_rate': [0.01, 0.1, 0.2],
                         'xgb__max_depth': [3, 5, 7],
                         'xgb__n_estimators': [100, 200]})

In [40]:
print("Best parameters found: ", grid.best_params_)

Best parameters found:  {'xgb__learning_rate': 0.2, 'xgb__max_depth': 7, 'xgb__n_estimators': 200}


In [41]:
best_model = grid.best_estimator_
y_pred_best = best_model.predict(x_test)
# Re-fit LabelEncoder on the original string labels to ensure le.classes_ are strings
# This is necessary because `le` was previously re-fitted on numerical data (y_train),
# causing le.classes_ to contain integers instead of strings.
le.fit(df['Emotion'])
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

              precision    recall  f1-score   support

       anger       0.91      0.95      0.93       392
        fear       0.96      0.90      0.93       416
         joy       0.95      0.96      0.96       380

    accuracy                           0.94      1188
   macro avg       0.94      0.94      0.94      1188
weighted avg       0.94      0.94      0.94      1188



In [45]:
import joblib

joblib.dump(best_model, 'model.joblib')
# Extract the fitted TfidfVectorizer from the best_model pipeline
tfidf_vectorizer_fitted = best_model.named_steps['tfidf']
joblib.dump(tfidf_vectorizer_fitted, 'tfidf_vectorizer_fitted.joblib')
joblib.dump(le, 'le.joblib')

['le.joblib']

## Instructions for your Flask Application

To correctly use the saved model components in your Flask application and avoid the `NotFittedError`, follow these steps:

1.  **Load the `best_model` (the Pipeline), the fitted `TfidfVectorizer`, and the `LabelEncoder`:**

    ```python
    import joblib
    import spacy

    # Load the trained pipeline (best_model)
    best_model = joblib.load('model.joblib')

    # Load the fitted TF-IDF vectorizer
    tfidf_vectorizer_fitted = joblib.load('tfidf_vectorizer_fitted.joblib')

    # Load the fitted LabelEncoder
    le = joblib.load('le.joblib')

    # Load your Spacy model for preprocessing
    nlp = spacy.load('en_core_web_sm')

    # Define your preprocessing function (same as in the notebook)
    def preprocess_text_for_flask(raw_texts):
        processed_texts = []
        for i in raw_texts:
            doc = nlp(i)
            processed_texts.append(' '.join([token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.is_space]))
        return processed_texts
    ```

2.  **Modify your prediction logic in your Flask app's `home` function:**

    Your original error trace pointed to `vectorizer.transform([cleaned])`. The `best_model` is a `Pipeline` that already contains the `TfidfVectorizer` and handles the transformation internally. You should use the `best_model` directly for prediction.

    ```python
    # Inside your Flask app's route function (e.g., @app.route('/predict', methods=['POST']))
    @app.route('/home') # Or whatever your route is
    def home():
        # ... (your existing code to get user input, e.g., 'text_input')

        # 1. Preprocess the input text using the same function as in the notebook
        #    Make sure preprocess_text_for_flask is defined and uses your loaded nlp object
        processed_input = preprocess_text_for_flask([text_input]) # text_input is the user's comment

        # 2. Predict using the loaded best_model (the pipeline handles vectorization internally)
        prediction_encoded = best_model.predict(processed_input)

        # 3. Inverse transform the prediction to get the original emotion label
        predicted_emotion = le.inverse_transform(prediction_encoded)

        # ... (rest of your Flask app logic)
        return f"The predicted emotion is: {predicted_emotion[0]}"
    ```

**Key takeaways:**

*   When using a `Pipeline`, you typically call `pipeline.predict()` directly on your raw (or preprocessed) input. The pipeline will internally apply all steps, including vectorization.
*   The `tfidf_vectorizer_fitted` was saved separately primarily for inspection or if you needed to use *only* the vectorizer for some other task, but for predictions with your trained model, the `best_model` pipeline is the way to go.
*   Ensure your Flask application has access to `model.joblib`, `tfidf_vectorizer_fitted.joblib`, `le.joblib`, and the Spacy model (`en_core_web_sm`).

In [43]:
x = input('Enter your Comment: ')
processed_comment = preprocessed_text([x])
print(le.inverse_transform(best_model.predict(processed_comment)))

Enter your Comment: iam happy
['anger']


In [44]:
import joblib

model = joblib.load("model.joblib")

samples = [
    "i were to ever get married i d have everything iam happy",
    "I feel so angry and frustrated",
    "I am scared of what will happen"
]

print(model.predict(samples))

[2 0 1]


In [38]:
df

,Comment,Emotion,Encoded_Emotions,processed_texts
0,i seriously hate one subject to death but now ...,fear,1,seriously hate subject death feel reluctant drop
1,im so full of life i feel appalled,anger,0,m life feel appalled
2,i sit here to write i start to dig out my feel...,fear,1,sit write start dig feeling think afraid accep...
3,ive been really angry with r and i feel like a...,joy,2,ve angry r feel like idiot trust place
4,i feel suspicious if there is no one outside l...,fear,1,feel suspicious outside like rapture happen
...,...,...,...,...
5932,i begun to feel distressed for you,fear,1,begin feel distressed
5933,i left feeling annoyed and angry thinking that...,anger,0,leave feel annoyed angry thinking center stupi...
5934,i were to ever get married i d have everything...,joy,2,marry d ready offer ve get club perfect good l...
5935,i feel reluctant in applying there because i w...,fear,1,feel reluctant apply want able find company kn...
